In [27]:
%%bash
export USE_SYMENGINE=1 

In [1]:


from functools import partial
from toolz import memoize, pipe, accumulate,groupby, compose,compose_left, merge
from collections import namedtuple
import pathlib,os
import typing as T
import pydrake
from pydrake.all import (
    MultibodyPlant,
    ModelInstanceIndex,
    Body, namedview,
    JacobianWrtVariable,
    Frame,
)


import symforce
symforce.set_symbolic_api("symengine")
symforce.set_log_level("warning")
import sympy as sp
from sympy.codegen import CodeBlock,Assignment
import symforce.symbolic as sf
import casadi as ca
from projects.refactor_mpb.symforce_casadi.casadi_config import CasadiConfig
from symforce.codegen.backends.pytorch import PyTorchConfig
from symforce.ops.storage_ops import StorageOps

import pydrake
from pydrake.symbolic import to_sympy as drake_to_sympy
from pydrake.all import (
    MultibodyPlant, MakeVectorVariable, JacobianWrtVariable, Frame,Body
)

from utils.my_sympy.misc import *
from utils.my_sympy.conversions import *
from utils.misc import *
from utils.my_drake.misc import *
from utils.math.quaternions import quaternion_dot_to_angular_velocity, hamiltonian_product
from utils.my_casadi.math import analytical_jacobian_to_geometric_matrix, quaternion_to_euler_angles, rot_matrix_to_quaternion


In [2]:
def replace_constants_pytorch(path):
    """
    Quick hax to replace constant tensors in a pytorch file with variables.
    """
    import re
    with open(path, 'r') as file:
        content = file.read()

    # Regular expression to match constant tensor declarations including fractions
    pattern = r"torch\.tensor\(([\d\.-]+|[\d\.-]+\ / [\d\.-]+), \*\*tensor_kwargs\)"
    matches = re.findall(pattern, content)

    # Evaluate the fractions and convert to float, create a map of unique constants to new variable names
    constant_to_variable = {}
    unique_constants = {}
    for match in set(matches):
        try:
            constant = float(eval(match))
            
            var_name = f"cte_{len(unique_constants) + 1}"
            if constant not in unique_constants:
                unique_constants[constant] = var_name
            constant_to_variable[match] = constant
        except Exception as e:
            print(f"Error evaluating match '{match}': {e}")
            continue

    # Replace the occurrences of constant tensor declarations with new variable names
    for match, constant in constant_to_variable.items():
        # constant_str = str(constant) if '/' not in str(constant) else f'({str(constant)})'
        var_name = unique_constants[constant]
        content = re.sub(
            pattern.replace(r'([\d\.-]+|[\d\.-]+\ / [\d\.-]+)', (match)),
            var_name,
            content
        )
    new_declarations = '\n'.join(f"    {var_name} = torch.tensor({constant}, **tensor_kwargs)" for constant, var_name in unique_constants.items())
    content = content.replace('# Input arrays','# Input arrays\n\n'+new_declarations)
    with open(path, 'w') as file:
        file.write(content)
def drake_matrix_to_sympy(matrix, memo):
    matrix = matrix.tolist()
    row_list = []
    for row in matrix:
        column_list = []
        for element in row:
            element = pydrake.symbolic.to_sympy(element,memo = memo)
            column_list.append(element)
        row_list.append(column_list)
    return sp.Matrix(row_list)
def quaternion_exponential(v:Iterable,eps:float = 1e-8) -> ca.MX:
    # https://theorangeduck.com/page/exponential-map-angle-axis-angular-velocity
    half_angle = ca.norm_2(v)
    result = ca.if_else(half_angle < eps,ca.vertcat(1,v[0],v[1],v[2]),ca.vertcat(ca.cos(half_angle),*(v*ca.sin(half_angle)/half_angle).nz))
    return result
def sympy_expression_to_symforce(sympy_expr, sympy_to_symforce_dict):
    # assert isinstance(sympy_expr,(sp.MutableSparseMatrix,sp.ImmutableSparseMatrix,sp.Matrix,sp.MutableDenseMatrix,sp.ImmutableMatrix,sp.ImmutableDenseMatrix)) , "im only really considering matrices right now so edit this to change to symforce"
    if isinstance(sympy_expr,(sp.MutableSparseMatrix,sp.ImmutableSparseMatrix,sp.Matrix,sp.MutableDenseMatrix,sp.ImmutableMatrix,sp.ImmutableDenseMatrix)):
        return sf.Matrix(sympy_expr.subs(sympy_to_symforce_dict).tolist())
    
    return sympy_expr

def angular_velocity_to_quat_dot_matrix(q):
    """
    Calculates the quaternion derivative from the angular velocity vector using CasADI.
    """
    H = sp.Matrix( [[-q[1], q[0], -q[3], q[2]],
                    [-q[2], q[3], q[0], -q[1]],
                    [-q[3], -q[2], q[1], q[0]]])
    return 1/2*H.T

def sympy_to_symforce(var,):
    if isinstance(var,(sp.Symbol,sp.Number)):
        return StorageOps.from_storage(sf.Symbol,[var])
    elif isinstance(var,(sp.MatrixSymbol)):
        raise ValueError("MatrixSymbols don't work with sympy->symengine conversion")
    elif isinstance(var,(sp.IndexedBase)):
        raise ValueError("IndexedBase don't work with sympy->symengine conversion")
    elif isinstance(var,(sp.Matrix,)):
        return sf.Matrix(var)
    else:
        try:
            return StorageOps.from_storage(sf.Expr,[var])
        except:
            print("Tried to convert",var, " with StorageOps.from_storage(sf.Expr,[var])")
            raise ValueError(f"Type {type(var)} not implemented in sympy_var_to_symforce")
def cse_then_print(expr,path):
    expr_cse = sp.cse(expr,order = 'none')
    temp_terms = [Assignment(*term) for term in (expr_cse)[0]]
    preamble = "from sympy import *"
    vars = sp.python(expr.free_symbols).split("\n")[:-1]
    intermediate_results = sp.pycode(CodeBlock(*temp_terms),fully_qualified_modules=False)
    output = sp.python(expr_cse[1][0]).split("\n")[-1]

    text = "\n".join([preamble] + vars + [intermediate_results] + [output])
    with open(path,'w') as f:
        f.write(text)
        
class MultiBodyPlantWrapper:
    """
    TODO: 
    - Dynamics
    - if using BSpline derivative, the derivatives of the orientation of free bodies are quaternion derivatives, not angular velocities
        - do something using the q_dot_to_angular_velocity function
    - maybe inherit from multibodyplant instead of having it as a member
    - FIX: having to call function(casadi_variable.nz) instead of function(casadi_variable)
    - calc_euler_angles_for_free_body(q,model_instance,'xyz')
    - conversion between jacobian types pg 24
        - https://ethz.ch/content/dam/ethz/special-interest/mavt/robotics-n-intelligent-systems/rsl-dam/documents/RobotDynamics2016/RD2016script.pdf
        - https://math.stackexchange.com/questions/3887845/mapping-from-quaternion-jacobian-to-geometric-jacobian
            quaternion one might me incorrect
    """
# context = plant.CreateDefaultContext()
# plant.SetPositions(context, [1]*14)
# q = ca.SX.sym('x',14,1)
# T = casadi_plant.calc_frame_pose_in_frame(q,robot_1_link_7,world_frame,clean_small_coeffs=1e-2)
# T2 = casadi_plant.calc_frame_pose_in_frame(q,world_frame,robot_1_link_6,clean_small_coeffs=1e-2)
# # T = T@T2
# point = np.array([0,0,0]).reshape(-1,1)
# R = T[0:3,0:3]
# t = (T@np.array([1,2,3,1]).reshape(-1,1))[0:3]
# # t = (T2@ca.vertcat(T[0:3,3],1))[0:3]
# Js_c = ca.jacobian(R,q)
# Jt_c = ca.jacobian(t,q)

# q_dot = ca.SX.sym('dx',14,1)
# v = ca.SX.sym('dx',13,1)
# W = (Js_c@q_dot).reshape((3,3))@R.T
# w_1 = W[2,1]
# w_2 = W[0,2]
# w_3 = W[1,0]
# rot = ca.vertcat(w_1,w_2,w_3)
# t = ca.vertcat(T2[0:3,0:3]@rot,(T2[0:3,0:3]@Jt_c@q_dot))
# # t = ca.vertcat(T2[0:3,0:3]@rot,Jt_c@q_dot)

# F = ca.Function('F',[q,q_dot],[ca.cse(t)])
# print(F([1]*14,[1]*14))
# print(plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kQDot,robot_1_link_7,[1,2,3],world_frame,robot_1_link_6)@np.array([1]*14))
    def __init__(self, plant: MultibodyPlant,cse=True, temp_folder = None) -> None:
        self.plant = plant
        if temp_folder is None:
            self.TEMP_FOLDER = pathlib.Path(os.getcwd()) / 'temp'
        else:
            self.TEMP_FOLDER = pathlib.Path(temp_folder)
        self.TEMP_FOLDER.mkdir(exist_ok=True,parents=True)
        
        self.mapping = {'ImmutableDenseMatrix': ca.blockcat,
                        'MutableDenseMatrix': ca.blockcat,
                        'Abs': ca.fabs
                        }
        
        self.cse = cse
        
        self.sym_plant = plant.ToSymbolic()
        self.sym_context = self.sym_plant.CreateDefaultContext()
        self.sym_world_frame = self.sym_plant.world_frame()
        self.drake_position_variables = MakeVectorVariable(
            self.plant.num_positions(), name='x')
        self.drake_velocity_variables = MakeVectorVariable(
            self.plant.num_velocities(), name='dx')
        self.sym_plant.SetPositions(self.sym_context, self.drake_position_variables)
        self.sym_plant.SetVelocities(self.sym_context, self.drake_velocity_variables)
        
        self.pos_vel_memo = merge({q.get_id(): sp.Symbol(f'q_{i}') for i,q in enumerate(self.drake_position_variables)}, {v.get_id(): sp.Symbol(f'v_{i}') for i,v in enumerate(self.drake_velocity_variables)})
        
        self.sympy_to_symforce = {v: sf.Symbol(v.name)  for k, v in self.pos_vel_memo.items()}

        self.sympy_velocity_variables = [drake_to_sympy(var, memo = self.pos_vel_memo) for var in self.drake_velocity_variables]
        self.sympy_position_variables = [drake_to_sympy(var, memo = self.pos_vel_memo) for var in self.drake_position_variables]
        
        self.free_bodies = get_all_free_bodies(self.plant)
        
        # self.forward_euler = self.make_forward_integration_euler_function()
        self.plant_named_view = make_namedview_positions(plant,'')
        self.position_upper_limits = self.plant.GetPositionUpperLimits()
        self.position_lower_limits = self.plant.GetPositionLowerLimits()
    
    def get_free_body_position_with_euler_angles(self, q:Iterable, model_instance:ModelInstanceIndex, order:str):
        model_instances_of_free_bodies = [body.model_instance() for body in self.free_bodies]
        assert model_instance in model_instances_of_free_bodies, f"Model instance {model_instance} not in {model_instances_of_free_bodies}"
        current_position_object = self.get_positions_from_array(model_instance,q)
        current_euler_angles = quaternion_to_euler_angles(current_position_object[0:4],order,extrinsic=False)
        current_translation = current_position_object[4:]
        current_position_object = ca.vertcat(current_euler_angles,current_translation)
        return current_position_object
    def named_vector(self, name:str,vector:Iterable):
        """
        Gets a named view of the vector `vector` with name `name`.
        """
        if isinstance(vector,(ca.MX,ca.SX,ca.DM)):
            return self.plant_named_view(name,vector.nz)
        return self.plant_named_view(name,vector)
    def set_position_upper_limits(self, position_upper_limits:Iterable):
        self.position_upper_limits = position_upper_limits
    def set_position_lower_limits(self, position_lower_limits:Iterable):
        self.position_lower_limits = position_lower_limits
        
    def get_joints_midpoints(self):
        lower_limit = self.position_upper_limits
        upper_limit = self.position_lower_limits
        if isinstance(lower_limit, (tuple,list)):
            lower_limit = np.array(lower_limit)
        if isinstance(upper_limit, (tuple,list)):
            upper_limit = np.array(upper_limit)    
        q_middle = (lower_limit+upper_limit)/2
        return q_middle
    
    def get_positions_from_array(self, model_instance:ModelInstanceIndex, q:Iterable) -> Iterable:
        """
        Extract elements from the array q at indices that align with the generalized positions of `model_instance` within the whole plant position vector." 
        
        I.e: `return q[appropriate_indices]`
        
        Generic version of `plant.GetPositionsFromArray(model_instance,q)` that works with CasADi variables.
        """
        
        x = list(range(0,self.num_positions()))
        return q[self.plant.GetPositionsFromArray(model_instance,x).astype(int)]
    def set_positions_in_array(self, model_instance:ModelInstanceIndex, q:Iterable, q_instance:Iterable) -> Iterable:
        """
        Updates the positions in the array `q` based on the positions associated with `model_instance`. 
        
        This method takes indices corresponding to `model_instance` within the plant, and replaces 
        the values at these indices in `q` with the values from `q_instance`.
        
        I.e: `q[appropriate_indices] = q_instance`
        
        Generic version of `plant.SetPositionsInArray(model_instance,q_instance,q)` that works with CasADi variables.
        """
        x = list(range(0,self.num_positions()))
        q[self.plant.GetPositionsFromArray(model_instance,x).astype(int)] = q_instance
        return q
    
    def get_velocities_from_array(self, model_instance:ModelInstanceIndex, q:Iterable) -> Iterable:
        """
        Extract elements from the array q at indices that align with the velocities of `model_instance` within the whole plant velocity vector." 
        
        I.e: `return v[appropriate_indices]`
        
        Generic version of `plant.GetVelocitiesFromArray(model_instance,q)` that works with CasADi variables.
        
        The difference between this method and `get_positions_from_array` is that this method returns the velocities of the plant, 
        which has angular velocity instead of quaternion derivatives.
        """
        
        x = list(range(0,self.num_velocities()))
        return q[self.plant.GetVelocitiesFromArray(model_instance,x).astype(int)]
    def set_velocities_in_array(self, model_instance:ModelInstanceIndex, v:Iterable, v_instance:Iterable) -> Iterable:
        """
        Updates the positions in the array `v` based on the velocities associated with `model_instance`. 
        
        This method takes indices corresponding to `model_instance` within the plant, and replaces 
        the values at these indices in `v` with the values from `v_instance`.
        
        I.e: `v[appropriate_indices] = v_instance`
        
        Generic version of `plant.SetVelocitiesInArray(model_instance,q_instance,q)` that works with CasADi variables.
        """
        x = list(range(0,self.num_velocities()))
        v[self.plant.GetVelocitiesFromArray(model_instance,x).astype(int)] = v_instance
        return v
    def num_positions(self):
        return self.plant.num_positions()

    def num_velocities(self):
         return self.plant.num_velocities()
    
    def get_sym_frame(self, frame:Frame):
        return self.sym_plant.GetFrameByName(frame.name(),frame.model_instance())
    def get_function_or_throw(self, function_name, path: pathlib.Path = None, module = 'casadi'):
        if path is None: 
            path = self.TEMP_FOLDER
            
        path = path / module
        if not (path / f"{function_name}.py").exists():
            raise ValueError(f"Function {function_name} not found in {path}")
        modul = symforce.codegen.codegen_util.load_generated_package(function_name, path / f"{function_name}.py")
        if module == 'sympy':
            sympy_expr = modul.e
            return sympy_expr
        function = getattr(modul, function_name)
        return function
    def make_function_from_expression(self, function_name, expression, inputs, path: pathlib.Path = None, module = 'casadi'):
        if path is None: 
            path = self.TEMP_FOLDER
            
        path = path / module
        path.mkdir(exist_ok=True,parents=True)
        
        if module == 'sympy':
            cse_then_print(expression,str(path / f"{function_name}.py"))
            return expression
        if module == 'casadi':
            return self.sympy_to_casadi(function_name, expression, inputs, path)
        if module == 'pytorch':
            return self.sympy_to_pytorch(function_name, expression, inputs, path)
    def sympy_to_module(self, function_name, sympy_expr, inputs, module, path: pathlib.Path = None):
        if path is None: path = self.TEMP_FOLDER
        assert isinstance(inputs, (list,tuple)), "inputs must be a list or tuple of variables"
        assert module in ['casadi','pytorch'] , "module must be one of ['casadi','pytorch']"
        config = CasadiConfig() if module == 'casadi' else PyTorchConfig()
        
        codegen_input = symforce.values.Values()
        for i,var in enumerate(inputs):
            sf_var = sympy_to_symforce(var)
            codegen_input[f"_in{i}"] = sf_var
            
        sf_expr = sympy_to_symforce(sympy_expr)
            
        codegen_output = symforce.values.Values(out = sf_expr)
        
        codegen_obj = symforce.codegen.Codegen(
            inputs=codegen_input,
            outputs=codegen_output,
            config=config,
            name=function_name,
            return_key="out",
        )

        
        generated_data = codegen_obj.generate_function(str(path), skip_directory_nesting=True)
        if module == 'pytorch':
            replace_constants_pytorch(str(generated_data.function_dir / f"{function_name}.py"))
        gen_module = symforce.codegen.codegen_util.load_generated_package(
            function_name, generated_data.function_dir
        )
        function = getattr(gen_module, function_name)
        return function
    def sympy_to_pytorch(self, function_name, sympy_expr, inputs, path: pathlib.Path = None):
        return self.sympy_to_module(function_name, sympy_expr, inputs, 'pytorch', path)
    def sympy_to_casadi(self, function_name, sympy_expr, inputs, path: pathlib.Path = None):
        return self.sympy_to_module(function_name, sympy_expr, inputs, 'casadi', path)
    
    @memoize
    def get_frame_pose_in_frame_sympy(self, frame:Frame, frame_ref:Frame, clean_small_coeffs):
        sym_frame:Frame = self.get_sym_frame(frame)
        sym_frame_ref = self.get_sym_frame(frame_ref)
        
        pose = drake_matrix_to_sympy(sym_frame.CalcPose(self.sym_context,sym_frame_ref).GetAsMatrix4(),self.pos_vel_memo)
        if clean_small_coeffs > 0:
            small_numbers = set([e for e in pose.atoms(sp.Number) if abs(e) < clean_small_coeffs])
            d = {s: 0 for s in small_numbers}
            pose = pose.subs(d)
        return pose
    
    def get_frame_pose_in_frame_function(self, frame:Frame, frame_ref:Frame, clean_small_coeffs:float = -1, module = 'casadi'):
        assert module in ['casadi','pytorch','sympy','numpy'] , "module must be one of ['casadi','pytorch','sympy','numpy']"
        
        function_name = f"frame_pose_in_frame_{frame.name()}_{frame_ref.name()}"
        inputs = [sp.Matrix(self.sympy_position_variables)]
        path = self.TEMP_FOLDER / "frame_pose_in_frame"
        path.mkdir(exist_ok=True,)
        try:
            return self.get_function_or_throw(function_name, path = path, module = module)
        except:
            pass
        try:
            sympy_expression = self.get_function_or_throw(function_name,path = path, module = 'sympy')
        except:
            sympy_expression = self.get_frame_pose_in_frame_sympy(frame,frame_ref,clean_small_coeffs)
            inputs = [sp.Matrix(self.sympy_position_variables)]
            return self.make_function_from_expression(function_name, sympy_expression, inputs, path = path, module = module)

    
    def calc_frame_pose_in_frame(self, q :Iterable,frame:Frame, frame_expressed:Frame, clean_small_coeffs:float = -1, module = 'casadi'):
        """
            Calculate the pose of `frame` expressed `frame_ref` using `q` as the generalized positions of the plant.
            
            This means that the point `p` in `frame` is expressed in `frame_ref` as `p_ref = pose @ p`.
            
            If `clean_small_coeffs` is set to a positive number, coefficients of the resulting pose smaller than that number will be set to zero, simplifying the expression. (recommended: 1e-6)
        """
        # assert module in ['casadi','pytorch','sympy','numpy'] , "module must be one of ['casadi','pytorch','sympy','numpy']"
        # if module == 'casadi':
        if module == 'sympy':
            return self.get_frame_pose_in_frame_function(frame,frame_expressed,clean_small_coeffs, module = module).subs({i:q for i,q in zip(self.sympy_position_variables,q)})
        return self.get_frame_pose_in_frame_function(frame,frame_expressed,clean_small_coeffs, module = module)(q)
    
    @memoize
    def velocity_to_generalized_velocity_matrix_sympy(self):
        M = sp.Matrix.zeros(self.num_positions(),self.num_velocities())

        v_named = namedview('',self.plant.GetVelocityNames())([a for a in self.sympy_velocity_variables])
        q_named = namedview('',self.plant.GetPositionNames())([a for a in self.sympy_position_variables])
        count = 0
        for i,name in enumerate(self.plant.GetPositionNames()):
            last_el = name.split('_')[-1]
            name = name[:-len(last_el)-1]
            if last_el == 'qw':
                # v = 
                count += 1
                qw = eval("q_named" + "." +  name+ "_qw")
                qx = eval("q_named" + "." +  name+ "_qx")
                qy = eval("q_named" + "." +  name+ "_qy")
                qz = eval("q_named" + "." +  name+ "_qz")
                m = angular_velocity_to_quat_dot_matrix([qw,qx,qy,qz])
                M[i:i+4,i:i+3] = m
                pass
            elif last_el in ['qx','qy','qz']:
                
                continue
            else:
                
                M[i,i-count] = 1
        return M
    # @memoize
    def get_velocity_to_generalized_velocity_matrix_function(self, module = 'casadi'):
        assert module in ['casadi','pytorch','sympy'] , "module must be one of ['casadi','pytorch','sympy']"
        
        function_name = f"velocity_to_generalized_velocity_matrix"
        inputs = [sp.Matrix(self.sympy_position_variables)]
        path = self.TEMP_FOLDER / "velocity_to_generalized_velocity_matrix"
        path.mkdir(exist_ok=True,)
        try:
            return self.get_function_or_throw(function_name, path = path, module = module)
        except:
            pass
        try:
            sympy_expression = self.get_function_or_throw(function_name,path = path, module = 'sympy')
        except:
            sympy_expression = self.velocity_to_generalized_velocity_matrix_sympy()
            inputs = [sp.Matrix(self.sympy_position_variables)]
            return self.make_function_from_expression(function_name, sympy_expression, inputs, path = path, module = module)
    def calc_velocity_to_generalized_velocity_matrix(self, q :Iterable, module = 'casadi'):
        """
        Calculate the matrix that maps the velocities of the plant to the generalized velocities of the plant.
        """
        if module == 'sympy':
            return self.get_velocity_to_generalized_velocity_matrix_function(module = module).subs({i:q for i,q in zip(self.sympy_position_variables,q)})
        return self.get_velocity_to_generalized_velocity_matrix_function(module = module)(q)
    
    @memoize
    def get_spatial_velocity_jacobian_sympy(self, with_respect_to:JacobianWrtVariable,frame_of_point:Frame,frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs):
        T1 = self.get_frame_pose_in_frame_function(frame_of_point,frame_measured_in,clean_small_coeffs,module= 'sympy')
        T2 = self.get_frame_pose_in_frame_function(frame_measured_in,frame_expressed_in,clean_small_coeffs,module= 'sympy')
        point = sp.Matrix([[sp.Symbol('x')],[sp.Symbol('y')],[sp.Symbol('z')],[1.0]])
        R = T1[0:3,0:3]
        t = (T1@point)[0:3,0]
        J_R = R.reshape(9,1).jacobian(self.sympy_position_variables)
        J_t = t.jacobian(self.sympy_position_variables)
        for i in range(self.num_positions()):
            J_R[:,i] = (J_R[:,i].reshape(3,3)@R.T).reshape(9,1)
        J_R = J_R[2*3+1,:].col_join(J_R[0*3+2,:]).col_join(J_R[1*3+0,:])
        rotational_velocities = T2[0:3,0:3]@J_R
        translational_velocities = T2[0:3,0:3]@J_t
        J = (rotational_velocities.col_join(translational_velocities))
        if with_respect_to == JacobianWrtVariable.kV:
            M = self.get_velocity_to_generalized_velocity_matrix_function(module = 'sympy')
            J = J@M
        return J
    
    def get_spatial_velocity_jacobian_function(self, with_respect_to:JacobianWrtVariable,frame_of_point:Frame,frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs, module = 'casadi'):
        assert module in ['casadi','pytorch','sympy'] , "module must be one of ['casadi','pytorch','sympy']"
        function_name = f"spatial_velocity_jacobian_{with_respect_to.name}_{frame_of_point.name()}_{frame_measured_in.name()}_{frame_expressed_in.name()}"
        inputs = [sp.Matrix(self.sympy_position_variables), sp.Matrix([[sp.Symbol('x')],[sp.Symbol('y')],[sp.Symbol('z')]])]
        path = self.TEMP_FOLDER / "spatial_velocity_jacobian"
        path.mkdir(exist_ok=True,)
        try:
            return self.get_function_or_throw(function_name, path = path, module = module)
        except:
            pass
        try:
            sympy_expression = self.get_function_or_throw(function_name,path = path, module = 'sympy')
        except:
            sympy_expression = self.get_spatial_velocity_jacobian_sympy(with_respect_to, frame_of_point, frame_measured_in,frame_expressed_in,clean_small_coeffs)
            return self.make_function_from_expression(function_name, sympy_expression, inputs, path = path, module = module)
    def calc_spatial_velocity_jacobian(self, q,with_respect_to:JacobianWrtVariable,frame_of_point:Frame,point:Iterable,frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs, module = 'casadi'):
        """
            Calculate the pose of `frame` expressed `frame_ref` using `q` as the generalized positions of the plant.
            
            This means that the point `p` in `frame` is expressed in `frame_ref` as `p_ref = pose @ p`.
            
            If `clean_small_coeffs` is set to a positive number, coefficients of the resulting pose smaller than that number will be set to zero, simplifying the expression. (recommended: 1e-6)
        """
        # assert module in ['casadi','pytorch','sympy','numpy'] , "module must be one of ['casadi','pytorch','sympy','numpy']"
        # if module == 'casadi':
        if module == 'sympy':
            subs = merge({i:q for i,q in zip(self.sympy_position_variables,q)},
                        {sp.Symbol('x'):point[0],sp.Symbol('y'):point[1],sp.Symbol('z'):point[2]})
            return self.get_spatial_velocity_jacobian_function(with_respect_to,frame_of_point,frame_measured_in,frame_expressed_in, clean_small_coeffs, module = 'sympy').subs(subs)
        return self.get_spatial_velocity_jacobian_function(with_respect_to,frame_of_point,frame_measured_in,frame_expressed_in, clean_small_coeffs, module = module)(q,point)
    # def calc_frame_pose_in_frame_euler(self, q :Iterable,frame:Frame, frame_expressed:Frame, order:str,clean_small_coeffs:float = -1):
    #     """
    #         Calculate the pose of `frame` expressed `frame_ref` using `q` as the generalized positions of the plant.
            
    #         Returns the pose as an array `[psi,theta,phi,x,y,z]` where `psi,theta,phi` are the euler angles of the orientation of `frame` expressed in `frame_ref` and `x,y,z` are the translation of `frame` expressed in `frame_ref`.
            
    #         If `clean_small_coeffs` is set to a positive number, coefficients of the resulting pose smaller than that number will be set to zero, simplifying the expression. (recommended: 1e-6)
    #     """
        
    #     matrix = self.get_frame_pose_in_frame_function(frame,frame_expressed,clean_small_coeffs)(q)
    #     translation = matrix[0:3,3]
    #     quaternion = rot_matrix_to_quaternion(matrix[0:3,0:3])
    #     euler = quaternion_to_euler_angles(quaternion,order,extrinsic=False)
    #     return ca.vertcat(euler,translation)
        
    # def calc_squared_distance_between_points(self, q :Iterable,frame_1:Frame, point_1:Iterable, frame_2:Frame,point_2:Iterable, clean_small_coeffs:float = -1, jacobian = False) -> Union[ca.MX,Iterable]:

    #     """
    #     Calculate the distance between `point_1` in `frame_1` and `point_2` in `frame_2` using `q` as the generalized positions of the plant.
        
    #     If `clean_small_coeffs` is set to a positive number, coefficients of the resulting distance smaller than that number will be set to zero, simplifying the expression.
        
    #     If `jacobian` is set to `True`, the function will also return the jacobian of the distance with respect to `q`. Returns `tuple(distance,jacobian)`.
    #     """
    #     frame_1_in_2_pose = self.calc_frame_pose_in_frame(q,frame_1,frame_2,clean_small_coeffs=clean_small_coeffs)
    #     p1 = frame_1_in_2_pose[0:3,0:3] @ point_1 + frame_1_in_2_pose[0:3,3]
    #     p2 = point_2
    #     distance = ca.sumsqr(p1-p2)
    #     if jacobian:
    #         jacobian = ca.jacobian(distance,q)
    #         return distance,jacobian
    #     else:
    #         return distance
    # def calc_norm_2_distance_between_points(self, q :Iterable,frame_1:Frame, point_1:Iterable, frame_2:Frame,point_2:Iterable, clean_small_coeffs:float = -1, jacobian = False) -> Union[ca.MX,Iterable]:

    #     """
    #     Calculate the distance between `point_1` in `frame_1` and `point_2` in `frame_2` using `q` as the generalized positions of the plant.
        
    #     If `clean_small_coeffs` is set to a positive number, coefficients of the resulting distance smaller than that number will be set to zero, simplifying the expression.
        
    #     If `jacobian` is set to `True`, the function will also return the jacobian of the distance with respect to `q`. Returns `tuple(distance,jacobian)`.
    #     """
    #     frame_1_in_2_pose = self.calc_frame_pose_in_frame(q.nz,frame_1,frame_2,clean_small_coeffs=clean_small_coeffs)
    #     p1 = frame_1_in_2_pose[0:3,0:3] @ point_1 + frame_1_in_2_pose[0:3,3]
    #     p2 = point_2
    #     distance = ca.norm_2(p1-p2)
    #     if jacobian:
    #         jacobian = ca.jacobian(distance,q)
    #         return distance,jacobian
    #     else:
    #         return distance
    # def forward_integration_euler(self, q :Iterable, v :Iterable, dt:float) -> Iterable:
    #     """
    #     Integrates the plant forward in time using Euler integration. Appropriately handles quaternions by using the exponential map.
    #     """
    #     # assert len(q) == len(v)
    #     # this check doesn't work with casadi variables so comment out if you want to do symbolic integration
    #     # if len(q) != len(v):
    #     #     raise ValueError(f"Length of q ({len(q)}) and v ({len(v)}) must be the same (non generalized velocities not implemented)")
    #     return self.forward_euler(q,v,dt)
    

    # def make_forward_integration_euler_function(self):
    #     """
    #     Makes a function that integrates the plant forward in time using Euler integration. Appropriately handles quaternions by using the exponential map.
    #     """
    #     q = ca.SX.sym('q',self.num_positions())
    #     v = ca.SX.sym('v',self.num_velocities())
    #     dt = ca.SX.sym('dt')
        
        
    #     v_named = namedview('',self.plant.GetVelocityNames())([a for a in v.nz])
    #     q_named = namedview('',self.plant.GetPositionNames())([a for a in q.nz])
    #     q_new = []
    #     for name in self.plant.GetPositionNames():
    #         last_el = name.split('_')[-1]
    #         name = name[:-len(last_el)-1]
    #         if last_el == 'qw':
    #             # v = 
    #             qw = eval("q_named" + "." +  name+ "_qw")
    #             qx = eval("q_named" + "." +  name+ "_qx")
    #             qy = eval("q_named" + "." +  name+ "_qy")
    #             qz = eval("q_named" + "." +  name+ "_qz")
    #             wx = eval("v_named" + "." +  name+ "_wx")
    #             wy = eval("v_named" + "." +  name+ "_wy")
    #             wz = eval("v_named" + "." +  name+ "_wz")
                
    #             quat = ca.vertcat(qw,qx,qy,qz)
    #             w = ca.vertcat(wx,wy,wz)
    #             new_quaternions = hamiltonian_product(quat,quaternion_exponential(w/2*dt))
    #             q_new.append(new_quaternions)
    #             pass
    #         elif last_el in ['qx','qy','qz']:
                
    #             continue
    #         else:
                
    #             state = eval("q_named" + "." +  name + "_" + last_el)
    #             def get_v_name(last_el):
    #                 match last_el:
    #                     case 'q':
    #                         return 'w'
    #                     case 'x':
    #                         return 'vx'
    #                     case 'y': 
    #                         return 'vy'
    #                     case 'z':
    #                         return 'vz'
    #             v_name = get_v_name(last_el)
    #             v_ = eval("v_named" + "." +  name + "_" + v_name)
    #             q_new.append(state + v_*dt)
        
        
        
    #     f = ca.Function('forward_integration_euler',[q,v,dt],[ca.vertcat(*q_new)],{'cse':True})
    #     return f
    
    # @memoize
    # @staticmethod
    # def get_euler_dot_to_angular_velocity_matrix(order):
    #     psi, theta, phi = ca.SX.sym('psi'), ca.SX.sym('theta'), ca.SX.sym('phi')
    #     M = analytical_jacobian_to_geometric_matrix(order,[psi,theta,phi])
        
    #     return ca.Function('euler_dot_to_angular_velocity_matrix',[psi,theta,phi],[M],{'cse':True,'post_expand':True,})
    
    # def calc_euler_dot_to_angular_velocity_matrix(self, euler_angles:Iterable, order):
    #     psi, theta, phi = euler_angles[0], euler_angles[1], euler_angles[2]
    #     return CasadiMultiBodyPlantWrapper.get_euler_dot_to_angular_velocity_matrix(order)(psi,theta,phi)
    
    # @memoize 
    # @staticmethod
    # def get_angular_velocity_to_euler_dot_matrix(order):
    #     psi, theta, phi = ca.SX.sym('psi'), ca.SX.sym('theta'), ca.SX.sym('phi')
    #     M = analytical_jacobian_to_geometric_matrix(order,[psi,theta,phi])
        
    #     return ca.Function('angular_velocity_to_euler_dot_matrix',[psi,theta,phi],[M],{'cse':True,'post_expand':True,})
    # def calc_angular_velocity_to_euler_dot_matrix(self, euler_angles:Iterable, order):
    #     psi, theta, phi = euler_angles[0], euler_angles[1], euler_angles[2]
    #     return CasadiMultiBodyPlantWrapper.get_angular_velocity_to_euler_dot_matrix(order)(psi,theta,phi)
    
    
    # @memoize
    # def get_frame_relative_velocity_function(self, frame:Frame, frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs):
    #     sym_frame:Frame = self.sym_plant.GetFrameByName(frame.name(),frame.model_instance())
    #     sym_frame_measured_in = self.sym_plant.GetFrameByName(frame_measured_in.name(),frame_measured_in.model_instance())
    #     sym_frame_expressed_in = self.sym_plant.GetFrameByName(frame_expressed_in.name(),frame_expressed_in.model_instance())
    #     velocity = sym_frame.CalcSpatialVelocity(self.sym_context,sym_frame_measured_in,sym_frame_expressed_in)
    #     translational_velocity = velocity.translational().tolist()
    #     rotational_velocity = velocity.rotational().tolist()
    #     velocity = [pydrake.symbolic.to_sympy(element) for element in rotational_velocity + translational_velocity]
    #     velocity = sp.Matrix(velocity).reshape(6,1)
    #     if clean_small_coeffs > 0:
    #         small_numbers = set([e for e in velocity.atoms(sympy.Number) if abs(e) < clean_small_coeffs])
    #         d = {s: 0 for s in small_numbers}
    #         velocity = velocity.subs(d)
    #     f = sp.lambdify([self.sympy_position_variables,self.sympy_velocity_variables],velocity,modules=[self.mapping,ca],cse=self.cse)
    #     return f
    # def calc_frame_relative_velocity(self, q :Iterable, v :Iterable,frame:Frame, frame_measured_in:Frame,frame_expressed_in:Frame, clean_small_coeffs:float = -1):
    #     """
    #         Calculate the velocity of `frame` relative to `frame_measured_in` expressed in `frame_expressed_in` using `q` as the generalized positions of the plant and `v` as the velocities of the plant.
            
    #         Note: If there are quaternion derivatives in `v` you have to convert them to angular velocities first using `quaternion_dot_to_angular_velocity` for now.
            
    #         If `clean_small_coeffs` is set to a positive number, coefficients of the resulting velocity smaller than that number will be set to zero, simplifying the expression.
    #     """
        
    #     return self.get_frame_relative_velocity_function(frame,frame_measured_in,frame_expressed_in,clean_small_coeffs)(q,v)


In [135]:

# Add new variable declarations at the beginning of the file
# content = f"{new_declarations}\n\n{content}"
# content

In [146]:
modul = symforce.codegen.codegen_util.load_generated_package("1", "/workspaces/toys/projects/thesis/temp/franka_diff_co/obstacle_distances_function/pytorch/obstacle_distances_function.py")
function_1 = getattr(modul, 'obstacle_distances_function')
modul = symforce.codegen.codegen_util.load_generated_package("12", "/workspaces/toys/projects/thesis/temp/franka_diff_co/obstacle_distances_function/pytorch/obstacle_distances_function copy.py")
function_2 = getattr(modul, 'obstacle_distances_function')
import torch
x = torch.tensor([2.]*21,)
function_1 = torch.compile((function_1))
function_1(x)
function_1(x)
function_1(x)
%timeit function_1(x)
%timeit function_2(x)
# torch.all(function_2(x) == function_1(x))

1.8 ms ± 464 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
12.3 ms ± 912 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [149]:
x = torch.tensor([2.]*21,dtype=torch.float16)
%timeit function_1(x)

1.76 ms ± 280 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [118]:
text =     """_out = _broadcast_and_stack(
        [
            torch.pow(
                _tmp0
                + torch.pow(
                    _tmp1 + torch.tensor(0.04, **tensor_kwargs), torch.tensor(2, **tensor_kwargs)
                )
                + torch.pow(
                    _tmp2 + torch.tensor(0.02, **tensor_kwargs), torch.tensor(2, **tensor_kwargs)
                    _tmp2 + torch.tensor(0.5, **tensor_kwargs), torch.tensor(2, **tensor_kwargs)
                ),
                torch.tensor(1 / 2, **tensor_kwargs),"""

{'2': 2.0, '0.5': 0.5, '0.04': 0.04, '1 / 2': 0.5, '0.02': 0.02}

In [3]:
from pydrake.all import (DiagramBuilder,AddMultibodyPlantSceneGraph,Parser,FindResourceOrThrow)
builder = DiagramBuilder()

plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.001)

ObjectFactory.box_object(plant=plant,scene_graph=scene_graph,**ObjectFactory.default_box_arguments())
franka_robot_urdf = FindResourceOrThrow("drake/manipulation/models/franka_description/urdf/panda_arm.urdf")
parser = Parser(plant)
(robot_1,) = Parser(plant).AddModels(franka_robot_urdf)
plant.RenameModelInstance(robot_1,"robot_1")
world_frame = plant.world_frame()

plant.WeldFrames(world_frame, plant.GetFrameByName('panda_link0',model_instance=robot_1), RigidTransform())

plant.Finalize()
casadi_plant = MultiBodyPlantWrapper(plant,cse = True)

In [4]:
plant_named_view = make_namedview_positions(plant,'')
context = plant.CreateDefaultContext()
plant.SetPositions(context, [1]*14)
# model instances
robot_1 = plant.GetModelInstanceByName("robot_1")
world_frame= plant.world_frame()
robot_1_link_7 = plant.GetFrameByName("panda_link7",robot_1)
robot_1_link_6 = plant.GetFrameByName("panda_link6",robot_1)
obstacle_frame = plant.GetFrameByName("box")
q = ca.SX.sym('dx',14,1)

f = casadi_plant.get_frame_pose_in_frame_function(obstacle_frame,world_frame,1e-6)

casadi_plant.calc_frame_pose_in_frame(q,obstacle_frame,world_frame,1e-6, module = 'casadi')
print(casadi_plant.calc_frame_pose_in_frame([1]*14,obstacle_frame,world_frame,1e-6, module = 'casadi'))

print(obstacle_frame.CalcPoseInWorld(context).GetAsMatrix4())
f_vel_gen = casadi_plant.get_velocity_to_generalized_velocity_matrix_function(module = 'pytorch')
import torch
print(f_vel_gen(torch.tensor([0]*7 + [1,0,0,0] + [0.,0,0])))
f_vel_gen = casadi_plant.get_velocity_to_generalized_velocity_matrix_function(module = 'casadi')
print(f_vel_gen([0]*7 + [1,0,0,0] + [0.,0,0]))
casadi_plant.calc_velocity_to_generalized_velocity_matrix([0]*7 + [1,0,0,0] + [0.,0,0], module = 'sympy')

codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code, you should set epsilon to either a symbol
    (recommended) or a small numerical value like `sf.numeric_epsilon`.  You should do
    this before importing any other code from symforce, e.g. with

        import symforce
        symforce.set_epsilon_to_symbol()

    or

        import symforce
        symforce.set_epsilon_to_number()

    For more information on use of epsilon to prevent singularities, take a look at the
    Epsilon Tutorial: https://symforce.org/tutorials/epsilon_tutorial.html



codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code, you should set epsilon to either a symbol
    (recommended) or a small numerical value like `sf.numeric_epsilon`.  You should do
    this before importing any other code from symforce, e.g. with

        import symforce
        symforce.set_epsilon_to_symbol()

    or

        import symforce
        symforce.set_epsilon_to_number()

    For more information on use of epsilon to prevent singularities, take a look at the
    Epsilon Tutorial: https://symforce.org/tutorials/epsilon_tutorial.html




[[0, 0, 1, 1], 
 [1, 0, 0, 1], 
 [0, 1, 0, 1], 
 [0, 0, 0, 1]]
[[0. 0. 1. 1.]
 [1. 0. 0. 1.]
 [0. 1. 0. 1.]
 [0. 0. 0. 1.]]


codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code, you should set epsilon to either a symbol
    (recommended) or a small numerical value like `sf.numeric_epsilon`.  You should do
    this before importing any other code from symforce, e.g. with

        import symforce
        symforce.set_epsilon_to_symbol()

    or

        import symforce
        symforce.set_epsilon_to_number()

    For more information on use of epsilon to prevent singularities, take a look at the
    Epsilon Tutorial: https://symforce.org/tutorials/epsilon_tutorial.html



tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, -0.0000, -0.0000,
         -0.0000, 0.0000, 0.0000, 0.0000],
        [0.00

Matrix([
[1, 0, 0, 0, 0, 0, 0,   0,   0,   0, 0, 0, 0],
[0, 1, 0, 0, 0, 0, 0,   0,   0,   0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0,   0,   0,   0, 0, 0, 0],
[0, 0, 0, 1, 0, 0, 0,   0,   0,   0, 0, 0, 0],
[0, 0, 0, 0, 1, 0, 0,   0,   0,   0, 0, 0, 0],
[0, 0, 0, 0, 0, 1, 0,   0,   0,   0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 1,   0,   0,   0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0,   0,   0,   0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0.5,   0,   0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0,   0, 0.5,   0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0,   0,   0, 0.5, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0,   0,   0,   0, 1, 0, 0],
[0, 0, 0, 0, 0, 0, 0,   0,   0,   0, 0, 1, 0],
[0, 0, 0, 0, 0, 0, 0,   0,   0,   0, 0, 0, 1]])

In [5]:
f_torch = casadi_plant.get_frame_pose_in_frame_function(obstacle_frame,world_frame,-1, module = 'pytorch')
import torch
print(f_torch(torch.tensor([1.]*14)))

codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code, you should set epsilon to either a symbol
    (recommended) or a small numerical value like `sf.numeric_epsilon`.  You should do
    this before importing any other code from symforce, e.g. with

        import symforce
        symforce.set_epsilon_to_symbol()

    or

        import symforce
        symforce.set_epsilon_to_number()

    For more information on use of epsilon to prevent singularities, take a look at the
    Epsilon Tutorial: https://symforce.org/tutorials/epsilon_tutorial.html



tensor([[0., 0., 1., 1.],
        [1., 0., 0., 1.],
        [0., 1., 0., 1.],
        [0., 0., 0., 1.]])


In [33]:
# q = ca.SX.sym('x',21,1)
# Js_1 = casadi_plant.calc_spatial_velocity_jacobian(q.nz,JacobianWrtVariable.kQDot, robot_1_link_7,[0,0,0],world_frame,world_frame,clean_small_coeffs=1e-6)
context = plant.CreateDefaultContext()
plant.SetPositions(context, [1]*14)
# print(plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kQDot,robot_1_link_7,[0,0,0],world_frame,world_frame))

q = ca.SX.sym('x',14,1)
T = casadi_plant.calc_frame_pose_in_frame(q,robot_1_link_7,world_frame,clean_small_coeffs=1e-2)
R = T[0:3,0:3]
t = T[0:3,3]
Js_c = ca.jacobian(R,q)
Jt_c = ca.jacobian(t,q)

q_dot = ca.SX.sym('dx',14,1)
v = ca.SX.sym('dx',13,1)
W = (Js_c@q_dot).reshape((3,3))@R.T
w_1 = W[2,1]
w_2 = W[0,2]
w_3 = W[1,0]
F = ca.Function('F',[q,q_dot],[ca.cse(ca.vertcat(w_1,w_2,w_3,Jt_c@q_dot))])
print(F([1]*14,[1]*14))
print(plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kQDot,robot_1_link_7,[0,0,0],world_frame,world_frame)@np.array([1]*14))

M = casadi_plant.calc_velocity_to_generalized_velocity_matrix(q, module = 'casadi')
W = (Js_c@(M)@v).reshape((3,3))@R.T
w_1 = W[2,1]
w_2 = W[0,2]
w_3 = W[1,0]
F = ca.Function('F',[q,v],[ca.cse(ca.vertcat(w_1,w_2,w_3,Jt_c@M@v))])
print(F([1]*14,[1]*13))
plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kV,robot_1_link_7,[0,0,0],world_frame,world_frame)@np.array([1]*13)




[0.632251, 2.49147, 1.53198, 0.348814, 0.259253, -0.681025]
[ 0.63225079  2.49146622  1.53197965  0.34881353  0.25925316 -0.68102484]
[0.632251, 2.49147, 1.53198, 0.348814, 0.259253, -0.681025]


array([ 0.63225079,  2.49146622,  1.53197965,  0.34881353,  0.25925316,
       -0.68102484])

In [41]:
context = plant.CreateDefaultContext()
plant.SetPositions(context, [1]*14)
q = ca.SX.sym('x',14,1)
T = casadi_plant.calc_frame_pose_in_frame(q,obstacle_frame,world_frame,clean_small_coeffs=1e-2)
T2 = casadi_plant.calc_frame_pose_in_frame(q,world_frame,robot_1_link_6,clean_small_coeffs=1e-2)
# T = T@T2
point = np.array([0,0,0]).reshape(-1,1)
R = T[0:3,0:3]
t = (T@np.array([1,2,3,1]).reshape(-1,1))[0:3]
# t = (T2@ca.vertcat(T[0:3,3],1))[0:3]
Js_c = ca.jacobian(R,q)
Jt_c = ca.jacobian(t,q)

q_dot = ca.SX.sym('dx',14,1)
v = ca.SX.sym('dx',13,1)
W = (Js_c@q_dot).reshape((3,3))@R.T
w_1 = W[2,1]
w_2 = W[0,2]
w_3 = W[1,0]
rot = ca.vertcat(w_1,w_2,w_3)
t = ca.vertcat(T2[0:3,0:3]@rot,(T2[0:3,0:3]@Jt_c@q_dot))
# t = ca.vertcat(T2[0:3,0:3]@rot,Jt_c@q_dot)

F = ca.Function('F',[q,q_dot],[ca.cse(t)])
print(F([1]*14,[1]*14))
print(plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kQDot,obstacle_frame,[1,2,3],world_frame,robot_1_link_6)@np.array([1]*14))

# t

[0, 0, 0, 1.32263, 0.117682, 1.11211]
[ 0.00000000e+00 -6.93889390e-17  1.11022302e-16  1.32263362e+00
  1.17682177e-01  1.11211115e+00]


In [35]:
context = plant.CreateDefaultContext()
plant.SetPositions(context, [1]*14)
q = ca.SX.sym('x',14,1)
T = casadi_plant.calc_frame_pose_in_frame(q,robot_1_link_7,world_frame,clean_small_coeffs=1e-2)
T2 = casadi_plant.calc_frame_pose_in_frame(q,world_frame,robot_1_link_6,clean_small_coeffs=1e-2)
# T = T@T2
point = np.array([0,0,0]).reshape(-1,1)
R = T[0:3,0:3]
t = (T@np.array([1,2,3,1]).reshape(-1,1))[0:3]
# t = (T2@ca.vertcat(T[0:3,3],1))[0:3]
Js_c = ca.jacobian(R,q)
Jt_c = ca.jacobian(t,q)
for i in range(14):
    Js_c[:,i] = (Js_c[:,i].reshape((3,3))@R.T).reshape((-1,1))
q_dot = ca.SX.sym('dx',14,1)
v = ca.SX.sym('dx',13,1)
# W = (Js_c@q_dot).reshape((3,3))
# W = (Js_c@q_dot).reshape((3,3))@R.T
# w_1 = W[2,1]
# w_2 = W[0,2]
# w_3 = W[1,0]
# rot = ca.vertcat(w_1,w_2,w_3)
rot = ca.vertcat(Js_c[1*3+2,:],Js_c[2*3+0,:],Js_c[0*3+1,:])
t = ca.vertcat(T2[0:3,0:3]@rot@q_dot,(T2[0:3,0:3]@Jt_c@q_dot))
# t = ca.vertcat(T2[0:3,0:3]@rot,Jt_c@q_dot)

F = ca.Function('F',[q,q_dot],[ca.cse(t)])
print(F([1]*14,[1]*14))
print(plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kQDot,robot_1_link_7,[1,2,3],world_frame,robot_1_link_6)@np.array([1]*14))

[1.82018, -1.04608, 2.13232, 3.86259, -5.61114, -6.13284]
[ 1.82017731 -1.04608018  2.13231739  3.86258627 -5.6111408  -6.13283613]


In [6]:
frame_of_point = robot_1_link_7
frame_measured_in = obstacle_frame
frame_expressed_in = robot_1_link_6
frame_expressed_in = world_frame
self = casadi_plant
clean_small_coeffs = 1e-6


# casadi_plant.get_spatial_velocity_jacobian_function = get_spatial_velocity_jacobian_function
# casadi_plant.calc_spatial_velocity_jacobian = calc_spatial_velocity_jacobian
# casadi_plant.get_spatial_velocity_jacobian_sympy = get_spatial_velocity_jacobian_sympy
J = casadi_plant.get_spatial_velocity_jacobian_sympy(JacobianWrtVariable.kQDot,frame_of_point,frame_measured_in,frame_expressed_in,clean_small_coeffs)
# J_v = get_spatial_velocity_jacobian_sympy(casadi_plant,JacobianWrtVariable.kV,frame_of_point,frame_measured_in,frame_expressed_in,clean_small_coeffs)
# J_l = sp.lambdify([self.sympy_position_variables,[sp.Symbol('x'),sp.Symbol('y'),sp.Symbol('z')]],J,cse=True)
# J_v_l = sp.lambdify([self.sympy_position_variables,[sp.Symbol('x'),sp.Symbol('y'),sp.Symbol('z')]],J_v,cse=True)
# with_respect_to:JacobianWrtVariable,frame_of_point:Frame,frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs, module = 'casadi'
J_v_casadi = casadi_plant.get_spatial_velocity_jacobian_function(JacobianWrtVariable.kV,frame_of_point,frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'casadi')
J_casadi = casadi_plant.get_spatial_velocity_jacobian_function(JacobianWrtVariable.kQDot,frame_of_point,frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'casadi')
J_v_pytorch = casadi_plant.get_spatial_velocity_jacobian_function(JacobianWrtVariable.kV,frame_of_point,frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'pytorch')
J_pytorch = casadi_plant.get_spatial_velocity_jacobian_function(JacobianWrtVariable.kQDot,frame_of_point,frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'pytorch')


# sp.python(sp.cse(J),)

# with open('J.py','w') as f:
#     f.write(sp.python(J_cse,))
# J_cse[1][0].shape

codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code, you should set epsilon to either a symbol
    (recommended) or a small numerical value like `sf.numeric_epsilon`.  You should do
    this before importing any other code from symforce, e.g. with

        import symforce
        symforce.set_epsilon_to_symbol()

    or

        import symforce
        symforce.set_epsilon_to_number()

    For more information on use of epsilon to prevent singularities, take a look at the
    Epsilon Tutorial: https://symforce.org/tutorials/epsilon_tutorial.html

codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code

In [7]:
truth_v = plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kV,frame_of_point,[1,2,3],frame_measured_in,frame_expressed_in)
truth_q = plant.CalcJacobianSpatialVelocity(context,JacobianWrtVariable.kQDot,frame_of_point,[1,2,3],frame_measured_in,frame_expressed_in)
print(np.max((J_v_casadi([1]*7 + [1,1,1,1] + [1,1,1],[1,2,3]) - truth_v).full()))
print(np.max((J_casadi([1]*7 + [1,1,1,1] + [1,1,1],[1,2,3]) - truth_q).full()))
import torch
print(np.max((J_v_pytorch(torch.tensor([1]*7 + [1,1,1,1.] + [1,1,1]),torch.tensor([1,2,3.])) - truth_v).detach().numpy()))
print(np.max((J_pytorch(torch.tensor([1]*7 + [1,1,1,1] + [1,1,1.]),torch.tensor([1,2,3.])) - truth_q).detach().numpy()))

point = [1,2,3.]
q = [1.]*14
print(np.max(casadi_plant.calc_spatial_velocity_jacobian(q,JacobianWrtVariable.kV,frame_of_point,point,frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'casadi') - truth_v))
print(np.max(casadi_plant.calc_spatial_velocity_jacobian(q,JacobianWrtVariable.kQDot,frame_of_point,point,frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'casadi') - truth_q))
print(torch.max(casadi_plant.calc_spatial_velocity_jacobian(torch.tensor(q),JacobianWrtVariable.kV,frame_of_point,torch.tensor(point),frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'pytorch') - truth_v))
print(torch.max(casadi_plant.calc_spatial_velocity_jacobian(torch.tensor(q),JacobianWrtVariable.kQDot,frame_of_point,torch.tensor(point),frame_measured_in,frame_expressed_in,clean_small_coeffs, module = 'pytorch') - truth_q))

2.2200130622707093e-11
2.2200130622707093e-11
4.87766338985729e-07
4.87766338985729e-07
2.2200130622707093e-11
2.2200130622707093e-11
tensor(4.8777e-07, dtype=torch.float64)
tensor(4.8777e-07, dtype=torch.float64)


In [57]:
from pydrake.all import FindResourceOrThrow, Parser, AddMultibodyPlantSceneGraph, RigidTransform, FixedOffsetFrame, DiagramBuilder, StartMeshcat
try:
    meshcat
    print("meshcat already defined")
except NameError:
    meshcat = StartMeshcat()
builder = DiagramBuilder()
obstacles = [{'shape':'sphere','radius':0.1,'name':'obstacle_1'},
             {'shape':'sphere','radius':0.1,'name':'obstacle_2'}]
plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.001)

obstacle_frames= []
obstacle_bodies = []
for i, obstacle in enumerate(obstacles):
    if 'name' not in obstacle:
        obstacle['name'] = 'obstacle_{}'.format(str(i))
    # body,frame,_,_ = obstacle(plant=plant,scene_graph =scene_graph)
    body,frame,_,_ = ObjectFactory.object_from_name(**obstacle,plant=plant,scene_graph =scene_graph)
    # body:Body
    obstacle_bodies.append(body)
    obstacle_frame = plant.AddFrame(frame=FixedOffsetFrame(
    name="obstacle_{}_frame".format(obstacle['name']),
    P=plant.world_frame(),
    X_PF=RigidTransform.Identity()))
    obstacle_frames.append(obstacle_frame)
    # plant.WeldFrames(obstacle_frame, frame, RigidTransform([0.0,0.,0.]))
    
franka_robot_urdf = FindResourceOrThrow("drake/manipulation/models/franka_description/urdf/panda_arm.urdf")
parser = Parser(plant)
(robot_1,) = Parser(plant).AddModels(franka_robot_urdf)
plant.RenameModelInstance(robot_1,"robot_1")
world_frame = plant.world_frame()

plant.WeldFrames(world_frame, plant.GetFrameByName('panda_link0',model_instance=robot_1), RigidTransform())

plant.Finalize()

visualizer = add_default_meshcat_visualization(builder,meshcat)

diagram = builder.Build()
plant_wrapper = CasadiMultiBodyPlantWrapper(plant,temp_folder = pathlib.Path(os.getcwd()) / 'temp' / "franka_diff_co")
diagram = diagram
plant = plant
scene_graph = scene_graph
diagram_context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(diagram_context)
obstacle_frames = obstacle_frames
obstacle_bodies = obstacle_bodies
collision_geometries_robot = [plant.GetCollisionGeometriesForBody(plant.get_body(body_index)) for body_index in plant.GetBodyIndices(plant.GetModelInstanceByName("robot_1"))]
collision_geometries_robot = [item for sublist in collision_geometries_robot for item in sublist]
collision_geometries_obstacles = [plant.GetCollisionGeometriesForBody(body) for body in obstacle_bodies]
collision_geometries_obstacles = [item for sublist in collision_geometries_obstacles for item in sublist]

inspector = scene_graph.model_inspector()
robot_collision_transforms = []
for i, collision_geometry in enumerate(collision_geometries_robot):
    radius = inspector.GetShape(collision_geometry).radius()
    pose_in_frame = inspector.GetPoseInFrame(collision_geometry)
    col_frame = plant.GetBodyFromFrameId(inspector.GetFrameId(collision_geometry)).body_frame()
    sympy_transform = plant_wrapper.get_frame_pose_in_frame_function(col_frame,world_frame,1e-6, module = 'sympy')@pose_in_frame.GetAsMatrix4()
    robot_collision_transforms.append({'radius':radius,'transform':sympy_transform})
obstacle_collision_transforms = []
for i, collision_geometry in enumerate(collision_geometries_obstacles):
    radius = inspector.GetShape(collision_geometry).radius()
    pose_in_frame = inspector.GetPoseInFrame(collision_geometry)
    col_frame = plant.GetBodyFromFrameId(inspector.GetFrameId(collision_geometry)).body_frame()
    sympy_transform = plant_wrapper.get_frame_pose_in_frame_function(col_frame,world_frame,1e-6, module = 'sympy')@pose_in_frame.GetAsMatrix4()
    obstacle_collision_transforms.append({'radius':radius,'transform':sympy_transform})

distance_expressions = [ ]
for robot_collision in robot_collision_transforms:
    robot_radius = robot_collision['radius']
    robot_transform = robot_collision['transform']
    robot_translation = robot_transform[0:3,3]
    for obstacle_collision in obstacle_collision_transforms:
        obstacle_radius = obstacle_collision['radius']
        obstacle_transform = obstacle_collision['transform']
        obstacle_translation = obstacle_transform[0:3,3]
        diff = (robot_translation - obstacle_translation)
        distance = sp.sqrt(diff.T@diff)[0,0] - robot_radius - obstacle_radius
        distance_expressions.append(distance)
        
distance_matrix = sp.Matrix(distance_expressions)
pytorch_partial = partial(plant_wrapper.sympy_to_pytorch, sympy_expr = distance_matrix,function_name = "obstacle_distances_function", inputs = [sp.Matrix(plant_wrapper.sympy_position_variables)], path = plant_wrapper.TEMP_FOLDER / 'pytorch')
func_pytorch = plant_wrapper.get_function_or_make("obstacle_distances_function", pytorch_partial, module = 'pytorch', path = plant_wrapper.TEMP_FOLDER)
casadi_partial = partial(plant_wrapper.sympy_to_casadi, sympy_expr = distance_matrix,function_name = "obstacle_distances_function", inputs = [sp.Matrix(plant_wrapper.sympy_position_variables)], path = plant_wrapper.TEMP_FOLDER / "obstacle_distances_function" / 'casadi')
q = ca.SX.sym('x',21,1)
func_casadi = plant_wrapper.get_function_or_make("obstacle_distances_function", casadi_partial, module = 'casadi', path = plant_wrapper.TEMP_FOLDER / "obstacle_distances_function")
func_casadi = ca.Function('obstacle_distances_function',[q],[func_casadi(q)],{'cse':True,'jit':True,    'jit_options':
        {
        'flags': '-Ofast -march=native -shared -ffast-math -fno-finite-math-only -fopenmp',
        'linker_flags': '-fopenmp',
        'verbose':False,
        
        'cleanup':True,
        'temp_suffix':True,
        },})


meshcat already defined


In [58]:
%%bash
export OPENBLAS_NUM_THREADS=12
export OMP_NUM_THREADS=12

In [67]:
fcm = func_casadi.map(10000,'thread',1)
v = torch.randn(21,10000).numpy()
v = ca.DM([[1.]*21]*10000).T
%timeit fcm(v)
# fcm([1]*21)

11.3 ms ± 169 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [88]:
import torch
configuration = create_default_view(namedview('pos',plant.GetPositionNames()))
configuration.obstacle_1_obstacle_1_x = 0.6*0
configuration.obstacle_2_obstacle_2_x = 0.6*100
configuration.robot_1_panda_joint2_q = 2
# print(plant.num_positions())
c = configuration[:]
def f_drake():
    plant.SetPositions(plant_context,c)
    # diagram.ForcedPublish(diagram_context)
    query = scene_graph.get_query_output_port().Eval(
        scene_graph.GetMyContextFromRoot(diagram_context)
    )
    # obstacle_collision = [False]*len(obstacle_bodies)
    # self_collision = False
    distances = []
    for geo_robot in collision_geometries_robot:
        for i,geo_obstacle in enumerate(collision_geometries_obstacles):
            distance = query.ComputeSignedDistancePairClosestPoints(geo_robot,geo_obstacle).distance
            distances.append(distance)
func_pytorch_compile = torch.compile(func_pytorch,dynamic=True)

func_pytorch_compile(torch.tensor(c))
# func_compile = torch.compile(func)
# c_tensor = torch.tensor(c)
# func_compile(c_tensor)
# func_compile(c_tensor)
# func_compile(c_tensor)
# %timeit f_drake()
# %timeit func_compile(c_tensor)
# (func(torch.tensor(configuration[:])) - np.array(distances))
# torch.clip(func(torch.tensor(configuration))[::2],0,np.inf).numpy()

tensor([-1.1528e-01,  5.9820e+01, -8.1260e-02,  5.9830e+01, -8.1260e-02,
         5.9830e+01, -7.5147e-02,  5.9900e+01, -6.1511e-02,  5.9900e+01,
        -7.6334e-02,  5.9870e+01, -7.6334e-02,  5.9870e+01, -6.1511e-02,
         5.9900e+01, -4.0836e-02,  5.9930e+01, -5.1833e-02,  5.9930e+01,
        -4.0836e-02,  5.9930e+01, -2.5836e-02,  5.9960e+01, -4.7000e-02,
         5.9840e+01, -7.7000e-02,  5.9840e+01,  4.3000e-02,  5.9840e+01,
         1.3000e-02,  5.9840e+01, -1.7000e-02,  5.9840e+01,  1.1591e-01,
         5.9841e+01,  1.8134e-01,  5.9841e+01,  1.7483e-01,  5.9841e+01,
         1.7300e-01,  5.9841e+01,  1.7483e-01,  5.9841e+01,  1.8134e-01,
         5.9841e+01,  1.4453e-01,  5.9685e+01,  1.4281e-01,  5.9717e+01,
         1.5303e-01,  5.9777e+01,  1.5021e-01,  5.9653e+01,  1.6129e-01,
         5.9617e+01,  1.2445e-01,  5.9587e+01,  1.2915e-01,  5.9587e+01,
         1.4281e-01,  5.9597e+01,  1.2915e-01,  5.9587e+01,  1.2445e-01,
         5.9587e+01,  1.8136e-01,  5.9555e+01,  1.9

In [223]:
torch.linalg.norm(torch.tensor([[[1.,2,3.]]*2,[[2.,3,4.]]*2]).view(2,2,3),axis=2).mean(axis=-1)

tensor([3.7417, 5.3852])

In [181]:
tensor = torch.randn((100,21))
neg_distances = func_pytorch_compile(tensor) < 0
# torch.any(, dim = -1)
num_obstacles = 2
obstacle_1 = neg_distances[...,::num_obstacles]
obstacle_2 = neg_distances[...,1::num_obstacles]
# obstacle_n = neg_distances[...,n-1::num_obstacles]
# torch.any(obstacle_2,dim=-1)
# batch_size, total_distances = neg_distances.shape

In [182]:
num_obstacles = 2
neg_distances = func_pytorch_compile(tensor) < 0
# if neg_distances.dim() == 1:  # Non-batched input, [total_distances]
#     # Add a batch dimension to make it [1, total_distances]
#     neg_distances = neg_distances.unsqueeze(0)
obstacle_1 = neg_distances[...,::num_obstacles]
obstacle_2 = neg_distances[...,1::num_obstacles]
batch_size, total_distances = neg_distances.shape
num_points = total_distances // num_obstacles
reshaped_distances = neg_distances.view(batch_size, num_points, num_obstacles)
no_collision = torch.any(reshaped_distances, dim=1)

In [198]:
# np.argwhere(no_collision)
# no_collision[95,0]
# func_pytorch_compile(tensor)[95].
# reshaped_distances

tensor([[[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        [[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        [[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        ...,

        [[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        [[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, False]],

        [[False, False],
         [False, False],
         [False, False],
         ...,
         [False, False],
         [False, False],
         [False, 

In [205]:
neg_distances = func_pytorch_compile(tensor) < 0
if neg_distances.dim() == 1:  # Non-batched input, [total_distances]
    # Add a batch dimension to make it [1, total_distances]
    neg_distances = neg_distances.unsqueeze(0)

# Proceed as before
batch_size, total_distances = neg_distances.shape
num_points = total_distances // num_obstacles

# Reshape
reshaped_distances = neg_distances.view(batch_size, num_points, num_obstacles)

# Check collision
collision = torch.any(reshaped_distances, dim=1)

# If the input was originally non-batched, remove the added batch dimension from the output
if batch_size == 1:
    collision = collision.squeeze(0)
# np.argwhere(no_collision)
# collision

tensor([[False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, False],
        [False, 

In [174]:
(reshaped_distances[:,:,0] == obstacle_1).all()
(reshaped_distances[:,:,1] == obstacle_2).all()

tensor(True)

2.2200130622707093e-11
2.2200130622707093e-11
4.5879974752693897e-07
4.87766338985729e-07
2.2200130622707093e-11
2.2200130622707093e-11
tensor(4.5880e-07, dtype=torch.float64)
tensor(4.8777e-07, dtype=torch.float64)


In [39]:

print(ca.SX.sym('x',9).reshape((3,3)))
sp.Matrix([[Symbol(f'x_{i}')] for i in range(9)]).reshape(3,3)


[[x_0, x_3, x_6], 
 [x_1, x_4, x_7], 
 [x_2, x_5, x_8]]


Matrix([
[x_0, x_1, x_2],
[x_3, x_4, x_5],
[x_6, x_7, x_8]])

In [40]:
gdfsgdf
from toolz import do
x = sp.MatrixSymbol('x',2,2)
y = sp.MatrixSymbol('y',2,2)
x = sp.symbols('x1:5')
x = sp.Matrix(x).reshape(2,2)
y = sp.symbols('y1:5')
y = sp.Matrix(y).reshape(2,2)

# x[0,0] = 3
expr = (x + y)

expr = symforce.ops.storage_ops.StorageOps.from_storage(sf.Matrix(2,2),expr)
# sf_expr = sf.Matrix([expr.subs(d)])
# sf_expr = sf.Expr(expr.subs(d))
inputs = symforce.values.Values()
sf.Matrix
inputs["x"] = symforce.ops.storage_ops.StorageOps.from_storage(sf.Matrix(2,2),x)
inputs["y"] = symforce.ops.storage_ops.StorageOps.from_storage(sf.Matrix(2,2),y)


display(inputs)
outputs = symforce.values.Values(ddang=expr)

display(outputs)
double_pendulum = symforce.codegen.Codegen(
    inputs=inputs,
    outputs=outputs,
    config=CasadiConfig(),
    name="double_pendulum",
    return_key="ddang",
)
# sf.
double_pendulum_data = double_pendulum.generate_function(str('.'), skip_directory_nesting=True)


NameError: name 'gdfsgdf' is not defined

In [ ]:
# symforce.ops.ScalarLieGroupOps.from_storage(x,x)
# symforce.ops.impl.sequence_group_ops.SequenceStorageOps.storage_dim([1,2,3,4])
# symforce.ops.impl.sequence_group_ops.SequenceGroupOps.from_storage([sf.Expr]*4,x)
# x
g = sf.Symbol("g")
type((g+3))

In [ ]:
from projects.refactor_mpb.symforce_casadi.casadi_config import CasadiConfig
import casadi as ca
from symforce.codegen import Codegen
import symforce.symbolic as sf
import typing as T
import symforce
from symforce.values import Values


def drake_matrix_to_sympy(matrix, memo):
    matrix = matrix.tolist()
    row_list = []
    for row in matrix:
        column_list = []
        for element in row:
            element = pydrake.symbolic.to_sympy(element,memo = memo)
            column_list.append(element)
        row_list.append(column_list)
    return sp.Matrix(row_list)
# T = casadi_plant.calc_frame_pose_in_frame(q.nz,robot_1_link_7,world_frame,clean_small_coeffs=1e-2)
sym_frame = casadi_plant.sym_plant.GetFrameByName(robot_1_link_7.name(),robot_1_link_7.model_instance())
sym_frame_ref = casadi_plant.sym_plant.GetFrameByName(world_frame.name(),world_frame.model_instance())
# pose = 
pos_vel_memo = {q.get_id(): sp.Symbol(f'q_{i}') for i,q in enumerate(casadi_plant.drake_position_variables)}
        
# pose = [pydrake.symbolic.to_sympy(element,memo = pos_vel_memo) for row in pose for element in row ]
pose = drake_matrix_to_sympy(sym_frame.CalcPose(casadi_plant.sym_context,sym_frame_ref).GetAsMatrix4(),pos_vel_memo)
sympy_to_symforce = {v: sf.Symbol(v.name)  for k, v in pos_vel_memo.items()}
sympy_vars = list(pos_vel_memo.values())
clean_pose = []
for p in pose:
    if isinstance(p,(float,int)):
        clean_pose.append(p)
        continue
    small_numbers = set([e for e in p.atoms(sp.Number) if abs(e) < 1e-6])
    d = {s: 0 for s in small_numbers}
    p = p.subs(d)
    clean_pose.append(p)
# pose = sp.Matrix(clean_pose).reshape(4,4)
# sympy_to_symforce = {v: sf.Symbol(v.name)  for k, v in self.pos_vel_memo.items()}
# 
# f = lambdify([sympy_vars],pose,modules=[casadi_plant.mapping,ca],cse=casadi_plant.cse)

In [ ]:


import casadi as ca
from projects.refactor_mpb.symforce_casadi.casadi_config import CasadiConfig
# from symforce.codegen.Codegen import Codegen
import symforce.symbolic as sf
from symforce.codegen.backends.pytorch import PyTorchConfig

import typing as T
import symforce
from symforce.values import Values

symforce.set_symbolic_api("symengine")
symforce.set_log_level("warning")
import pathlib,os
TEMP_FOLDER = pathlib.Path(os.getcwd()) / 'temp'
TEMP_FOLDER.mkdir(exist_ok=True)
inputs = symforce.values.Values()
pose = sf.Matrix(pose.subs(sympy_to_symforce).tolist())
inputs["q"] = sf.Matrix(list(sympy_to_symforce.values()))


display(inputs)
outputs = symforce.values.Values(ddang=pose)

display(outputs)
double_pendulum = symforce.codegen.Codegen(
    inputs=inputs,
    outputs=outputs,
    config=CasadiConfig(),
    name="double_pendulum",
    return_key="ddang",
)
# sf.
double_pendulum_data = double_pendulum.generate_function(str(TEMP_FOLDER), skip_directory_nesting=True)
double_pendulum = symforce.codegen.Codegen(
    inputs=inputs,
    outputs=outputs,
    config=PyTorchConfig(),
    name="double_pendulum_torch",
    return_key="ddang",
)
# sf.
double_pendulum_data = double_pendulum.generate_function(str(TEMP_FOLDER), skip_directory_nesting=True)

gen_module = symforce.codegen.codegen_util.load_generated_package(
    "t", double_pendulum_data.function_dir
)